# Week 8-2 · DMP-02 — Introduction to Object-Oriented Programming
**Module:** Data Analysis & Modeling in Python · Instructor: Ashutosh Dave.

A new coding **paradigm**. So far you wrote *sequential / functional* code; production code (everything on
GitHub, every library you import) is written with **classes and objects**. Part 1 builds the concepts on a
`Student` class; Part 2 applies them by turning a moving-average back-test into a reusable
`BacktestingCrossover` class on real NIFTY data.

> Part 2 uses `yfinance` + `quantstats` in the lecture (neither installed here), so the class reads the
> shipped `nifty50_daily.csv` (962 bars, 2017→2020) and computes returns directly.

## 1. Dunders — the magic behind the operators
A **dunder** ("double underscore") is a built-in method like `__add__` or `__len__`. They look special but
behave like ordinary methods. `2 + 2` secretly calls `(2).__add__(2)`; `len(x)` calls `x.__len__()`.

In [1]:
print("2 + 2          =", 2 + 2)
print("(2).__add__(2) =", (2).__add__(2))     # same thing
lst = [10, 20, 30]
print("len(lst)        =", len(lst))
print("lst.__len__()   =", lst.__len__())      # same thing
import numpy as np
print("numpy version dunder:", np.__version__)

2 + 2          = 4
(2).__add__(2) = 4
len(lst)        = 3
lst.__len__()   = 3
numpy version dunder: 2.3.5


## 2. Everything in Python is an object
A *class* is a blueprint; an *object* (instance) is a specific thing built from it. `4.3` is an object of
class `float`; a list, a function — all objects. The special dunder **`__init__`** is the one OOP leans on.

In [2]:
a = 4.3
print("type(4.3):", type(a))            # <class 'float'>
print("4.3 is integer? ", a.is_integer())
print("4.0 is integer? ", (4.0).is_integer())

type(4.3): <class 'float'>
4.3 is integer?  False
4.0 is integer?  True


## 3. A first class: `Student` — class attributes & class methods
- `goal` defined in the class body is a **class attribute** (shared by all students).
- `@classmethod` links a method to the class; its first parameter is `cls` (the class itself).

In [3]:
class Student:
    goal = "to get educated"                 # class attribute (shared)

    @classmethod
    def take_lunch(cls):
        return "enjoying the lunch"

    @classmethod
    def update_goal(cls, new_goal):
        cls.goal = new_goal                  # changes it for the WHOLE class

john = Student()
mary = Student()
print("john.goal:", john.goal)
print("john.take_lunch():", john.take_lunch())
Student.update_goal("to get educated and get a job")
print("after update -> mary.goal:", mary.goal, "| class goal:", Student.goal)

john.goal: to get educated
john.take_lunch(): enjoying the lunch
after update -> mary.goal: to get educated and get a job | class goal: to get educated and get a job


## 4. Instance attributes & methods — `__init__` and `self`
Things that differ *per object* (name, age) live on the **instance**. `__init__` runs automatically when you
create an object; `self` represents that object. `self.name = name` creates an **instance attribute**.

In [4]:
class Student:
    goal = "to get educated"

    def __init__(self, name, age):           # constructor
        self.name = name                     # instance attribute
        self.age = age

    def update_age(self, new_age):           # instance method (first param self)
        self.age = new_age
        return f"new age for {self.name} is {self.age}"

    @staticmethod
    def academic_year():                     # no self/cls -> static method
        return "Academic year 2024-25"

john = Student("John Paul", 19)
print(john.name, "| age", john.age)
print(john.update_age(20))
print("static:", Student.academic_year())

John Paul | age 19
new age for John Paul is 20
static: Academic year 2024-25


**Three method types:**
| Kind | Decorator | First arg | Knows about |
|---|---|---|---|
| class method | `@classmethod` | `cls` | the class |
| instance method | *(none)* | `self` | the object |
| static method | `@staticmethod` | *(none)* | nothing — just grouped under the class |

## 5. Inheritance — parent (super) & child (sub) classes
A child class inherits everything from its parent and may override pieces. `SportyStudent(Student)` gets all
of `Student` for free; we only change the `goal`.

In [5]:
class SportyStudent(Student):
    goal = "to get educated and win some trophies"   # overrides parent

dave = SportyStudent("Dave", 17)
print("dave.name:", dave.name, "| dave.goal:", dave.goal)
print("inherited update_age:", dave.update_age(18))
print("isinstance(dave, SportyStudent):", isinstance(dave, SportyStudent))
print("isinstance(dave, Student)      :", isinstance(dave, Student))   # True via inheritance
print("issubclass(SportyStudent, Student):", issubclass(SportyStudent, Student))

dave.name: Dave | dave.goal: to get educated and win some trophies
inherited update_age: new age for Dave is 18
isinstance(dave, SportyStudent): True
isinstance(dave, Student)      : True
issubclass(SportyStudent, Student): True


### `__dict__` and the MRO (method resolution order)
`__dict__` lists an object's instance attributes; `.__mro__` is the family tree, always ending in `object`.

In [6]:
print("dave.__dict__:", dave.__dict__)
print("MRO:", [c.__name__ for c in SportyStudent.__mro__])

dave.__dict__: {'name': 'Dave', 'age': 18}
MRO: ['SportyStudent', 'Student', 'object']


## 6. `super()` and multiple inheritance
`super().__init__(...)` lets a child run its own `__init__` *and* the parent's. A child can inherit from
**several** parents; the MRO decides the search order.

In [7]:
class ParentOne:
    def __init__(self, age):
        self.age = age
        print(f"ParentOne __init__, age {age}")

class ParentTwo:
    def greet(self):
        print("hello from ParentTwo")

class Child(ParentOne, ParentTwo):
    def __init__(self, grade, age):
        self.grade = grade
        print(f"Child __init__, grade {grade}")
        super().__init__(age)                # also run ParentOne's __init__

jack = Child("A", 5)
jack.greet()
print("MRO:", [c.__name__ for c in Child.__mro__])

Child __init__, grade A
ParentOne __init__, age 5
hello from ParentTwo
MRO: ['Child', 'ParentOne', 'ParentTwo', 'object']


## 7. Built-ins are objects too
Check the MRO of a list, a numpy array, a DataFrame — each is a class with ancestors ending in `object`.
A DataFrame has many parents because it's a rich data structure.

In [8]:
import pandas as pd
print("list  MRO:", [c.__name__ for c in list.__mro__])
print("array MRO:", [c.__name__ for c in np.ndarray.__mro__])
df = pd.DataFrame({"x":[1,2,3]})
print("DataFrame MRO:", [c.__name__ for c in type(df).__mro__])

list  MRO: ['list', 'object']
array MRO: ['ndarray', 'object']
DataFrame MRO: ['DataFrame', 'NDFrame', 'PandasObject', 'DirNamesMixin', 'IndexingMixin', 'OpsMixin', 'object']


## Part 2 — Applying OOP: a `BacktestingCrossover` class
The same 7-step back-test (idea → data → indicators → signals → positions → returns → analysis), but wrapped
in a class. `__init__` stores the parameters and calls each step (an instance method) in order. One object =
one back-test. We trade an SMA-short / SMA-long crossover.

In [9]:
class BacktestingCrossover:
    def __init__(self, csv_path, ma_short, ma_long):
        self.csv_path = csv_path           # instance attributes
        self.ma_short = ma_short
        self.ma_long  = ma_long
        # run the pipeline automatically
        self.fetch_data()
        self.indicators()
        self.signals()
        self.positions()
        self.returns()

    def fetch_data(self):
        self.df = pd.read_csv(self.csv_path, index_col="Date",
                              parse_dates=True)[["Close"]].copy()

    def indicators(self):
        self.df["SMA_s"] = self.df["Close"].rolling(self.ma_short).mean()
        self.df["SMA_l"] = self.df["Close"].rolling(self.ma_long).mean()
        self.df = self.df.dropna()

    def signals(self):
        self.df["regime"] = np.where(self.df["SMA_s"] > self.df["SMA_l"], 1, -1)

    def positions(self):
        self.df["pos"] = self.df["regime"].shift(1)
        self.df = self.df.dropna()

    def returns(self):
        self.df["bh"]    = self.df["Close"].pct_change()
        self.df["strat"] = self.df["pos"] * self.df["bh"]
        self.total = (1 + self.df["strat"]).cumprod().iloc[-1] - 1
        print(f"SMA{self.ma_short}/{self.ma_long}: total return {self.total*100:.2f}%")
        return self.total

print("class defined")

class defined


### One object = one back-test. Compare two parameter sets.

In [10]:
path = "nifty50_daily.csv"
bt_10_20 = BacktestingCrossover(path, 10, 20)
bt_5_20  = BacktestingCrossover(path,  5, 20)
bh_total = (1 + bt_10_20.df["bh"]).cumprod().iloc[-1] - 1
print(f"\nBuy & Hold over same window: {bh_total*100:.2f}%")
print("Both back-tests share ONE blueprint; only the parameters differ.")

SMA10/20: total return 2.64%
SMA5/20: total return 5.62%

Buy & Hold over same window: 51.48%
Both back-tests share ONE blueprint; only the parameters differ.


### Equity curves for the two strategies

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,3.6))
for bt, c, lab in [(bt_10_20, "#7e22ce", "SMA10/20"), (bt_5_20, "#c084fc", "SMA5/20")]:
    eq = (1 + bt.df["strat"]).cumprod()
    ax.plot(eq.index, eq, color=c, lw=1.3, label=f"{lab} ({bt.total*100:.1f}%)")
bh_eq = (1 + bt_10_20.df["bh"]).cumprod()
ax.plot(bh_eq.index, bh_eq, color="#6b7280", lw=1.1, ls="--", label=f"Buy & Hold ({bh_total*100:.1f}%)")
ax.axhline(1, color="gray", lw=0.6)
ax.set_title("BacktestingCrossover — one class, two parameter sets")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("oop_preview.png", dpi=110)
print("equity curves drawn")

equity curves drawn


## Key takeaways
- **Dunders** (`__add__`, `__len__`, `__init__`) power the operators and built-ins; `2+2` is really `(2).__add__(2)`.
- A **class** is a blueprint; an **object/instance** is a specific thing. In Python *everything* is an object.
- **Class attribute** = shared (in the class body); **instance attribute** = per-object (set on `self` in `__init__`).
- **Three method types:** `@classmethod` (gets `cls`), instance method (gets `self`), `@staticmethod` (gets neither — just grouped under the class).
- **Inheritance:** a child class reuses a parent's code and overrides only what differs; `super().__init__()` runs the parent's constructor too; multiple inheritance follows the **MRO** (ends in `object`).
- `isinstance`, `issubclass`, `__dict__`, `__mro__` inspect the object/class graph.
- **OOP back-test:** wrapping the 7 steps in a `BacktestingCrossover` class makes each run one clean object — SMA5/20 beat SMA10/20 here, and creating a new back-test is a single line.